# Лабораторная работа № 4. GAN: генерация и восстановление изображений

В первой задаче вы обучите GAN генерировать одежду на Fashion-MNIST. Во второй - попробуете восстановить изображение после уменьшения разрешения, сравните обычную интерполяцию и компактную нейросеть.

## 0. Настройка окружения

Запустите в Colab или локально с `torch`, `torchvision`, `numpy` и `matplotlib`. В Colab при наличии GPU выберите *Среда выполнения -> Сменить среду выполнения -> GPU*. Датасеты загрузятся из `torchvision` при первом запуске; для Fashion-MNIST можно также использовать [датасет с kaggle](https://www.kaggle.com/datasets/zalando-research/fashionmnist).

В коде автоматически выбирается уменьшенный объём данных для CPU.

In [ ]:
import copy
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)
USE_GPU = device.type == "cuda"
BATCH_SIZE = 128 if USE_GPU else 64
NUM_WORKERS = 0  # одинаково запускается в Windows / Colab
print("Device:", device, "| batch size:", BATCH_SIZE)

# Задача 1. Генерация Fashion-MNIST при помощи GAN


### 1.1. Данные: загрузка, нормализация и просмотр


In [ ]:
# === РЕШЕНИЕ ===
import pandas as pd
import torchvision.utils as vutils
from IPython.display import display, Markdown

GAN_TRAIN_LIMIT = None if USE_GPU else 20000
GAN_EPOCHS = 30 if USE_GPU else 20
FASHION_CLASSES = ['футболка', 'брюки', 'пуловер', 'платье', 'пальто',
                   'сандалия', 'рубашка', 'кроссовок', 'сумка', 'ботинок']

gan_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
fashion_train = datasets.FashionMNIST(root=str(DATA_DIR), train=True, download=True, transform=gan_transform)
if GAN_TRAIN_LIMIT:
    gan_idx = torch.randperm(len(fashion_train), generator=torch.Generator().manual_seed(SEED))[:GAN_TRAIN_LIMIT]
    gan_data = Subset(fashion_train, gan_idx.tolist())
else:
    gan_data = fashion_train
gan_loader = DataLoader(gan_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                        num_workers=NUM_WORKERS, generator=torch.Generator().manual_seed(SEED))

images, labels = next(iter(gan_loader))
print(f"Картинок: {len(gan_data)}, итераций на эпоху: {len(gan_loader)}, эпох: {GAN_EPOCHS}")
print(f"batch картинок: {tuple(images.shape)} = (B, C, H, W), batch меток: {tuple(labels.shape)}")
print(f"диапазон значений: [{images.min():.2f}, {images.max():.2f}], среднее {images.mean():.3f}")


def to_unit(x):
    return ((x.detach().cpu() + 1) / 2).clamp(0, 1)


def show_grid(x, nrow=8, title=None, ax=None):
    grid = vutils.make_grid(to_unit(x), nrow=nrow, padding=2, pad_value=1.0)
    if ax is None:
        _, ax = plt.subplots(figsize=(nrow * 0.9, (len(x) + nrow - 1) // nrow * 0.95))
    ax.imshow(grid.permute(1, 2, 0))
    ax.axis('off')
    if title:
        ax.set_title(title)
    return ax


fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(to_unit(img)[0], cmap='gray')
    ax.set_title(FASHION_CLASSES[lab], fontsize=9)
    ax.axis('off')
fig.suptitle('Fashion-MNIST, примеры из batch')
plt.tight_layout()
plt.show()

Какова размерность batch? Для чего применяется нормализация в `[-1, 1]`? 

Batch картинок имеет форму `(B, 1, 28, 28)`: B = `BATCH_SIZE` (128 на GPU, 64 на CPU), 1 канал (оттенки серого), 28x28 пикселей. Метки имеют форму `(B,)`, для GAN они не используются.

Нормализация в `[-1, 1]` нужна, потому что генератор заканчивается `Tanh` и выдает значения в `[-1, 1]`. Реальные и сгенерированные картинки должны быть в одном диапазоне, иначе дискриминатор отличит их просто по диапазону значений. Кроме того, данные с центром около нуля удобнее для обучения сети.

### 1.2. Генератор и дискриминатор

Проверьте размерности входа/выхода, посчитайте число обучаемых параметров `G` и `D`, нарисуйте или опишите последовательность слоёв. Объясните, почему выходы `G` и `D` имеют разные функции активации.

In [ ]:
LATENT_DIM = 64

class Generator(nn.Module):
    def __init__(self, latent_dim=64, hidden=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden * 2), nn.ReLU(),
            nn.Linear(hidden * 2, 28 * 28), nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z).reshape(z.size(0), 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden // 2), nn.LeakyReLU(0.2),
            nn.Linear(hidden // 2, 1),  # без Sigmoid: выход - logit
        )

    def forward(self, x):
        return self.net(x)


# === РЕШЕНИЕ (продолжите)===
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


torch.manual_seed(SEED)
G = Generator(LATENT_DIM).to(device)
D = Discriminator().to(device)

with torch.no_grad():
    z_check = torch.randn(4, LATENT_DIM, device=device)
    x_fake = G(z_check)
    d_out = D(x_fake)
assert x_fake.shape == (4, 1, 28, 28) and d_out.shape == (4, 1)
print(f"z {tuple(z_check.shape)} -> G -> {tuple(x_fake.shape)}, значения в [{x_fake.min():.2f}, {x_fake.max():.2f}]")
print(f"x {tuple(x_fake.shape)} -> D -> {tuple(d_out.shape)} (logit)")
print(f"\nОбучаемых параметров: G = {count_params(G):,}, D = {count_params(D):,}")
for name, m in [('Generator', G), ('Discriminator', D)]:
    print(f"\n{name}:")
    for layer in m.net:
        print(f"  {str(layer):<58} {sum(p.numel() for p in layer.parameters()):>9,}")

Слои и параметры:
- G: z (64) -> Linear 64->256 + ReLU -> Linear 256->512 + ReLU -> Linear 512->784 + Tanh -> reshape в (1, 28, 28). Параметров 550 416.
- D: (1, 28, 28) -> Flatten (784) -> Linear 784->256 + LeakyReLU(0.2) -> Linear 256->128 + LeakyReLU(0.2) -> Linear 128->1. Параметров 233 985.

Почему разные выходные активации: G должен выдавать картинку в том же диапазоне, что и нормализованные данные, поэтому `Tanh` ограничивает выход отрезком [-1, 1]. D решает задачу "реальное или сгенерированное", его выход - logit, любое число. Сигмоиду применяет `BCEWithLogitsLoss` внутри, так численно устойчивее, чем `Sigmoid` + `BCELoss`. LeakyReLU в D пропускает градиент и при отрицательных входах, поэтому генератор получает сигнал для любых своих картинок.

### 1.3. Состязательное обучение

Одна итерация включает два отдельных шага:

1. **Обучаем `D`:** реальные изображения получают метку 1, изображения `G(z).detach()` - метку 0. `.detach()` не позволяет шагу дискриминатора обновлять генератор.
2. **Обучаем `G`:** создаём новые `G(z)`, хотим, чтобы `D(G(z))` соответствовал метке 1, и обновляем **только генератор**.


In [ ]:
GAN_LR = 2e-4
criterion_adv = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(G.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
fixed_z = torch.randn(16, LATENT_DIM, device=device)

# === РЕШЕНИЕ ===
SAVE_EVERY = max(1, GAN_EPOCHS // 10)
history = {'d_loss': [], 'g_loss': [], 'd_real': [], 'd_fake': []}
snapshots = {}
t0 = time.time()
for epoch in range(1, GAN_EPOCHS + 1):
    G.train()
    D.train()
    for real, _ in gan_loader:
        real = real.to(device)
        b = real.size(0)
        ones = torch.ones(b, 1, device=device)
        zeros = torch.zeros(b, 1, device=device)

        fake = G(torch.randn(b, LATENT_DIM, device=device)).detach()
        d_real, d_fake = D(real), D(fake)
        loss_d = criterion_adv(d_real, ones) + criterion_adv(d_fake, zeros)
        opt_d.zero_grad()
        loss_d.backward()
        opt_d.step()

        loss_g = criterion_adv(D(G(torch.randn(b, LATENT_DIM, device=device))), ones)
        opt_g.zero_grad()
        loss_g.backward()
        opt_g.step()

        history['d_loss'].append(loss_d.item())
        history['g_loss'].append(loss_g.item())
        history['d_real'].append(torch.sigmoid(d_real).mean().item())
        history['d_fake'].append(torch.sigmoid(d_fake).mean().item())

    if epoch == 1 or epoch % SAVE_EVERY == 0 or epoch == GAN_EPOCHS:
        G.eval()
        with torch.no_grad():
            snapshots[epoch] = G(fixed_z).cpu()
    n = len(gan_loader)
    print(f"эпоха {epoch:2d}/{GAN_EPOCHS} | D loss {np.mean(history['d_loss'][-n:]):.3f} | "
          f"G loss {np.mean(history['g_loss'][-n:]):.3f} | D(x) {np.mean(history['d_real'][-n:]):.2f} | "
          f"D(G(z)) {np.mean(history['d_fake'][-n:]):.2f} | {time.time() - t0:.0f} c")

### 1.4. Результаты и диагностика

**Подзадачи:**

1. Постройте два графика loss на одной системе координат; опишите, в какие моменты одна из сетей могла преобладать над другой.
2. Покажите изображения из первой, средней и последней сохранённых эпох (используйте один и тот же `fixed_z`). Какие изменения вы видите?
3. Сгенерируйте 16 новых изображений, сравните.


In [ ]:
# === РЕШЕНИЕ ===
def smooth(v, k=100):
    v = np.asarray(v)
    return np.convolve(v, np.ones(k) / k, mode='valid') if len(v) > k else v


iters_per_epoch = len(gan_loader)
ep_d = np.array(history['d_loss']).reshape(GAN_EPOCHS, -1).mean(1)
ep_g = np.array(history['g_loss']).reshape(GAN_EPOCHS, -1).mean(1)
ep_real = np.array(history['d_real']).reshape(GAN_EPOCHS, -1).mean(1)
ep_fake = np.array(history['d_fake']).reshape(GAN_EPOCHS, -1).mean(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
it = np.arange(len(smooth(history['d_loss']))) / iters_per_epoch + 100 / iters_per_epoch
axes[0].plot(it, smooth(history['d_loss']), color='#2a78d6', label='D loss (real + fake)')
axes[0].plot(it, smooth(history['g_loss']), color='#eb6834', label='G loss')
axes[0].axhline(2 * np.log(2), color='#2a78d6', ls=':', lw=1, label='D в равновесии: 2 ln 2 = 1.39')
axes[0].axhline(np.log(2), color='#eb6834', ls=':', lw=1, label='G в равновесии: ln 2 = 0.69')
axes[0].set_xlabel('эпоха')
axes[0].set_ylabel('BCE')
axes[0].set_title('Loss генератора и дискриминатора')
axes[0].legend(fontsize=8)
axes[1].plot(it, smooth(history['d_real']), color='#1baf7a', label='D(x), реальные')
axes[1].plot(it, smooth(history['d_fake']), color='#eda100', label='D(G(z)), сгенерированные')
axes[1].axhline(0.5, color='0.5', ls=':', lw=1)
axes[1].set_xlabel('эпоха')
axes[1].set_ylabel('вероятность "реальное"')
axes[1].set_ylim(0, 1)
axes[1].set_title('Ответы дискриминатора')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
print(pd.DataFrame({'D loss': ep_d, 'G loss': ep_g, 'D(x)': ep_real, 'D(G(z))': ep_fake},
                   index=pd.RangeIndex(1, GAN_EPOCHS + 1, name='эпоха')).round(3).to_string())

saved = sorted(snapshots)
chosen = [saved[0], saved[len(saved) // 2], saved[-1]]
fig, axes = plt.subplots(1, 3, figsize=(15, 5.3))
for ax, e in zip(axes, chosen):
    show_grid(snapshots[e], nrow=4, title=f'эпоха {e} (fixed_z)', ax=ax)
plt.tight_layout()
plt.show()

G.eval()
with torch.no_grad():
    z_new = torch.randn(16, LATENT_DIM, device=device, generator=torch.Generator(device=device).manual_seed(123))
    new_imgs = G(z_new).cpu()
fig, axes = plt.subplots(1, 2, figsize=(11, 5.6))
show_grid(new_imgs, nrow=4, title='16 новых картинок G(z)', ax=axes[0])
show_grid(images[:16], nrow=4, title='16 реальных картинок', ax=axes[1])
plt.tight_layout()
plt.show()

torch.manual_seed(SEED)
fm_clf = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10)).to(device)
opt_c = torch.optim.Adam(fm_clf.parameters(), lr=1e-3)
for _ in range(2):
    for xb, yb in gan_loader:
        loss = F.cross_entropy(fm_clf(xb.to(device)), yb.to(device))
        opt_c.zero_grad()
        loss.backward()
        opt_c.step()
fm_clf.eval()
fashion_test = datasets.FashionMNIST(root=str(DATA_DIR), train=False, download=True, transform=gan_transform)
x_ft = torch.stack([fashion_test[i][0] for i in range(2000)])
y_ft = torch.tensor([fashion_test[i][1] for i in range(2000)])
with torch.no_grad():
    clf_acc = (fm_clf(x_ft.to(device)).argmax(1).cpu() == y_ft).float().mean().item()
    real_conf = fm_clf(x_ft.to(device)).softmax(1).max(1).values.mean().item()
    gen_many = G(torch.randn(1000, LATENT_DIM, device=device))
    gen_probs = fm_clf(gen_many).softmax(1).cpu()
gen_cls = gen_probs.argmax(1).numpy()
gen_conf = gen_probs.max(1).values.mean().item()
class_share = pd.Series(np.bincount(gen_cls, minlength=10) / len(gen_cls), index=FASHION_CLASSES)


def hf_noise(x):
    c = x[:, :, 1:-1, 1:-1]
    lap = 4 * c - x[:, :, :-2, 1:-1] - x[:, :, 2:, 1:-1] - x[:, :, 1:-1, :-2] - x[:, :, 1:-1, 2:]
    return lap.abs().mean().item()


real_many = torch.stack([gan_data[i][0] for i in range(1000)])
div_gen = torch.pdist(gen_many[:500].cpu().flatten(1)).mean().item()
div_real = torch.pdist(real_many[:500].flatten(1)).mean().item()
noise_gen, noise_real = hf_noise(gen_many.cpu()), hf_noise(real_many)
print(f"Классификатор: accuracy на 2000 тестовых картинках {clf_acc:.3f}")
print("Доли классов среди 1000 сгенерированных картинок:")
print(class_share.round(3).to_string())
print(f"Средняя уверенность классификатора: реальные {real_conf:.3f}, сгенерированные {gen_conf:.3f}")
print(f"Среднее попарное L2-расстояние: сгенерированные {div_gen:.2f}, реальные {div_real:.2f}")
print(f"Мелкий шум (модуль лапласиана): сгенерированные {noise_gen:.3f}, реальные {noise_real:.3f}")

#### Ответ по 1.4

In [ ]:
def epoch_ranges(epochs):
    if not epochs:
        return ''
    parts, start, prev = [], epochs[0], epochs[0]
    for e in epochs[1:] + [None]:
        if e is not None and e == prev + 1:
            prev = e
            continue
        parts.append(f"{start}-{prev}" if prev > start else f"{start}")
        if e is not None:
            start = prev = e
    return ', '.join(parts)


LN2 = np.log(2)
d_strong = [e + 1 for e in range(GAN_EPOCHS) if ep_d[e] < 2 * LN2 - 0.3 and ep_fake[e] < 0.35]
g_strong = [e + 1 for e in range(GAN_EPOCHS) if ep_d[e] > 2 * LN2]
peaks = [e + 1 for e in range(1, GAN_EPOCHS - 1) if ep_g[e] > ep_g[e - 1] and ep_g[e] > ep_g[e + 1] and ep_g[e] > ep_g.mean()]
third = max(1, GAN_EPOCHS // 3)

text = ["1) Loss. Если D не отличает реальные картинки от сгенерированных (D = 0.5), то D loss = 2 ln 2 = 1.39, "
        "а G loss = ln 2 = 0.69. Чем ниже D loss и выше G loss, тем сильнее дискриминатор."]
if d_strong:
    text.append(f"- Дискриминатор заметно сильнее генератора (D loss < 1.09, D(G(z)) < 0.35) в эпохи {epoch_ranges(d_strong)}. "
                f"Максимум G loss {ep_g.max():.2f} в эпоху {int(ep_g.argmax()) + 1}.")
if peaks:
    text.append(f"- Локальные пики G loss в эпохи {epoch_ranges(peaks)}: в эти моменты D вырывался вперед, "
                "после этого G подстраивался и его loss снижался.")
if g_strong:
    text.append(f"- Генератор брал верх (D loss выше 1.39) в эпохи {epoch_ranges(g_strong)}.")
else:
    text.append("- Эпох, где генератор явно сильнее дискриминатора (D loss выше 1.39), не было.")
text.append(f"- В последней эпохе D loss {ep_d[-1]:.2f}, G loss {ep_g[-1]:.2f}, D(x) {ep_real[-1]:.2f}, "
            f"D(G(z)) {ep_fake[-1]:.2f}. Среднее D(G(z)) в первой трети обучения {ep_fake[:third].mean():.2f}, "
            f"в последней {ep_fake[-third:].mean():.2f}"
            + (", сети постепенно приближаются к равновесию." if ep_fake[-third:].mean() > ep_fake[:third].mean() + 0.03
               else "."))
text.append("- Loss в GAN показывает баланс двух сетей, а не качество картинок, поэтому качество смотрим по картинкам.")

text.append("\n2) Картинки на fixed_z.")
with torch.no_grad():
    snap_cls = {e: fm_clf(snapshots[e].to(device)).argmax(1).cpu().numpy() for e in chosen}
for e in chosen:
    x = snapshots[e]
    text.append(f"- Эпоха {e}: среднее расстояние между 16 картинками {torch.pdist(x.flatten(1)).mean():.2f}, "
                f"мелкий шум {hf_noise(x):.3f}, разных классов по классификатору: {len(set(snap_cls[e]))}.")
first, last = chosen[0], chosen[-1]
div_first = torch.pdist(snapshots[first].flatten(1)).mean().item()
div_last = torch.pdist(snapshots[last].flatten(1)).mean().item()
sat = {e: ((snapshots[e].abs() > 0.96).float().mean().item()) for e in chosen}
s = f"- От эпохи {first} к эпохе {last} "
s += ("картинки стали заметно разнообразнее" if div_last > 1.2 * div_first else "разнообразие почти не изменилось")
s += (f", доля почти черных или белых пикселей выросла с {sat[first]:.2f} до {sat[last]:.2f} (картинки контрастнее)."
      if sat[last] > sat[first] + 0.05 else ".")
same = int((snap_cls[chosen[1]] == snap_cls[last]).sum())
s += f" У {same} из 16 векторов класс со средней эпохи сохранился до последней"
s += ", то есть один и тот же z обычно закрепляется за одним типом предмета." if same >= 10 else "."
text.append(s)

top = class_share.sort_values(ascending=False)
present = top[top >= 0.02]
missing = [c for c in FASHION_CLASSES if class_share[c] < 0.02]
text.append("\n3) 16 новых картинок и 1000 сгенерированных.")
text.append(f"- Классификатор (accuracy {clf_acc:.3f} на тесте) на 1000 сгенерированных картинках: классов с долей "
            f"от 2% - {len(present)} из 10 (" + ', '.join(f"{c} {v:.0%}" for c, v in present.items()) + ")."
            + (f" Почти не встречаются: {', '.join(missing)}." if missing else ""))
text.append(f"- Средняя уверенность классификатора: на реальных {real_conf:.2f}, на сгенерированных {gen_conf:.2f}.")
text.append(f"- Разнообразие (среднее попарное расстояние): сгенерированные {div_gen:.1f}, реальные {div_real:.1f}"
            + (". Разнообразие заметно ниже реального, есть признаки mode collapse." if div_gen < 0.85 * div_real
               else ". Разнообразие близко к реальному, сильного mode collapse нет."))
if noise_gen > noise_real:
    text.append(f"- Мелкого шума у сгенерированных картинок в {noise_gen / noise_real:.1f} раза больше, чем у реальных: "
                "полносвязный генератор без сверток плохо передает гладкие области и мелкие детали.")
else:
    text.append(f"- Мелкого шума у сгенерированных картинок не больше, чем у реальных "
                f"({noise_gen:.3f} против {noise_real:.3f}).")
display(Markdown('\n'.join(text)))

### 1.5. Исследование латентного пространства (без повторного обучения)

Возьмите два случайных вектора `z_a` и `z_b`, постройте между ними 9 точек линейной интерполяции и сгенерируйте изображения. Меняется ли объект плавно? Сохраняется ли разнообразие?

Сравните изображения для `z`, `0.5z` и `1.5z`. Эти входы не обязаны соответствовать распределению обучения. Отметьте, меняется ли внешний вид и можно ли делать выводы о качестве по такой визуализации.

In [ ]:
# === РЕШЕНИЕ ===
G.eval()
gen = torch.Generator(device=device).manual_seed(7)
alphas = torch.linspace(0, 1, 9, device=device)


def slerp(a, b, t):
    omega = torch.acos((a / a.norm() * b / b.norm()).sum().clamp(-1, 1))
    return (torch.sin((1 - t) * omega) * a + torch.sin(t * omega) * b) / torch.sin(omega)


rows, norms, steps, row_cls = [], [], [], []
with torch.no_grad():
    for _ in range(3):
        z_a = torch.randn(LATENT_DIM, device=device, generator=gen)
        z_b = torch.randn(LATENT_DIM, device=device, generator=gen)
        zs = torch.stack([(1 - a) * z_a + a * z_b for a in alphas])
        imgs = G(zs).cpu()
        rows.append(imgs)
        norms.append(zs.norm(dim=1).cpu())
        steps.append((imgs[1:] - imgs[:-1]).flatten(1).norm(dim=1))
        row_cls.append(fm_clf(imgs.to(device)).argmax(1).cpu().numpy())
    rows.append(G(torch.stack([slerp(z_a, z_b, a) for a in alphas])).cpu())

fig, axes = plt.subplots(len(rows), 1, figsize=(10, 1.35 * len(rows) + 0.6))
for k, (ax, r) in enumerate(zip(axes, rows)):
    show_grid(r, nrow=9, ax=ax)
    ax.set_title('пара 3, сферическая интерполяция (slerp)' if k == 3 else f'пара {k + 1}, линейная интерполяция', fontsize=9)
plt.tight_layout()
plt.show()
print("||z|| вдоль линейной интерполяции (пара 1):", np.round(norms[0].numpy(), 1),
      f"| для N(0, I) типичная норма около sqrt({LATENT_DIM}) = {LATENT_DIM ** 0.5:.1f}")

with torch.no_grad():
    z8 = torch.randn(8, LATENT_DIM, device=device, generator=gen)
    scaled = {s: G(s * z8).cpu() for s in (0.5, 1.0, 1.5)}
    scaled_cls = {s: fm_clf(x.to(device)).argmax(1).cpu().numpy() for s, x in scaled.items()}
fig, axes = plt.subplots(3, 1, figsize=(9, 4.4))
for ax, (s, imgs) in zip(axes, scaled.items()):
    show_grid(imgs, nrow=8, ax=ax)
    ax.set_title(f'{s} * z  (||z|| около {s * LATENT_DIM ** 0.5:.0f})', fontsize=9)
plt.tight_layout()
plt.show()
scale_stats = {}
for s, imgs in scaled.items():
    x = to_unit(imgs)
    scale_stats[s] = dict(bright=x.mean().item(), sat=((x < 0.02) | (x > 0.98)).float().mean().item(),
                          spread=x.flatten(1).std(0).mean().item())
    print(f"{s}*z: средняя яркость {scale_stats[s]['bright']:.3f}, доля почти черных или белых пикселей "
          f"{scale_stats[s]['sat']:.3f}, разброс между картинками {scale_stats[s]['spread']:.3f}")

#### Ответ по 1.5

In [ ]:
step_ratio = max((st.max() / st.mean()).item() for st in steps)
same_cls = int(((scaled_cls[0.5] == scaled_cls[1.0]) & (scaled_cls[1.5] == scaled_cls[1.0])).sum())
bright = np.array([[to_unit(r[k]).mean().item() for k in range(9)] for r in rows[:3]])
mid_darker = bright[:, 3:6].mean() < bright[:, [0, 8]].mean() - 0.01
text = [f"- Интерполяция: самый большой шаг между соседними кадрами больше среднего в {step_ratio:.1f} раза"
        + (", резких скачков нет, предмет меняется плавно." if step_ratio < 2.5 else ", в одном месте есть заметный скачок."),
        "- По классификатору переходы такие: "
        + '; '.join(f"пара {k + 1}: {FASHION_CLASSES[rc[0]]} -> {FASHION_CLASSES[rc[-1]]}" for k, rc in enumerate(row_cls))
        + (". Промежуточные кадры - смеси форм двух предметов." if any(rc[0] != rc[-1] for rc in row_cls) else "."),
        f"- Норма z при линейной интерполяции падает с {norms[0][0]:.1f} до {norms[0].min():.1f} в середине пути, "
        f"а для N(0, I) почти все векторы имеют норму около {LATENT_DIM ** 0.5:.0f}. Середина пути лежит в "
        "области, которую генератор при обучении почти не видел"
        + (", и картинки там бледнее (средняя яркость в середине "
           f"{bright[:, 3:6].mean():.3f} против {bright[:, [0, 8]].mean():.3f} на концах)." if mid_darker else ".")
        + " Сферическая интерполяция сохраняет норму.",
        f"- Масштабирование: доля почти черных или белых пикселей при 0.5z - {scale_stats[0.5]['sat']:.2f}, "
        f"при z - {scale_stats[1.0]['sat']:.2f}, при 1.5z - {scale_stats[1.5]['sat']:.2f}. "
        + ("С ростом нормы картинки становятся контрастнее. " if scale_stats[1.5]['sat'] > scale_stats[0.5]['sat'] else "")
        + f"Класс предмета (по классификатору) одинаков для всех трех масштабов у {same_cls} из 8 векторов"
        + (": направление z в основном задает, что нарисовано." if same_cls >= 6 else "."),
        "- Делать выводы о качестве по 0.5z и 1.5z нельзя: такие векторы лежат вне распределения, на котором учился "
        "генератор. Качество оценивают на z из N(0, I), а для количественной оценки используют метрики вроде FID."]
display(Markdown('\n'.join(text)))

# Задача 2. Восстановление изображения после уменьшения разрешения

В исходной лабораторной предлагалось повышать качество изображений с помощью автоэнкодера и GAN, используя сложные фотографии примерно от 300 x 300. Здесь оставим саму идею и **снизим вычислительные требования**: начнём с цветных **CIFAR-10 (32 x 32)**. Из каждого изображения получим вход **16 x 16**; цель - восстановить исходное **32 x 32**.

Сравнить `bicubic` (без обучения) с компактным автоэнкодером, научить сеть предсказывать полноразмерное изображение и оценить её на test-наборе. **Дополнительно:** добавить к восстановлению условный дискриминатор (*conditional GAN / image-to-image*) и сравнить с автоэнкодером. Таким образом первоначальное сравнение двух архитектур сохранено в расширенном варианте, но не перегружает базовый уровень.

### 2.1. Данные

Используйте обучающий раздел CIFAR-10, часть отделите под validation, а официальный test **не используйте до итоговой проверки**. Пары `(low, high)` формируются из одного изображения, поэтому их нельзя случайно разбивать независимо друг от друга. Более высокое разрешение (например, 128 x 128) оставьте для отдельного проекта.

**Выполните:** визуализируйте триплеты: исходное `32x32` -> уменьшенное `16x16` -> bicubic-увеличение до `32x32`. Объясните, какую информацию модель принципиально не может знать точно.

In [ ]:
from torch.utils.data import Dataset

RESTORE_TRAIN_LIMIT = 10000 if USE_GPU else 4000
RESTORE_EPOCHS = 6 if USE_GPU else 4

class LowHighPairs(Dataset):
    def __init__(self, original_dataset, indices=None):
        self.original = original_dataset
        self.indices = list(range(len(original_dataset))) if indices is None else list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        high, _ = self.original[self.indices[idx]]  # tensor (3, 32, 32) в [0, 1]
        low = F.interpolate(high.unsqueeze(0), size=(16, 16), mode="area").squeeze(0)
        return low, high

cifar_train = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True,
                               transform=transforms.ToTensor())
permutation = torch.randperm(len(cifar_train), generator=torch.Generator().manual_seed(SEED + 1))
selected = permutation[:RESTORE_TRAIN_LIMIT].tolist()
cut = int(len(selected) * 0.85)
rest_train = LowHighPairs(cifar_train, selected[:cut])
rest_val = LowHighPairs(cifar_train, selected[cut:])
restore_train_loader = DataLoader(rest_train, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS)
restore_val_loader = DataLoader(rest_val, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS)
print("Train:", len(rest_train), "| Validation:", len(rest_val),
      "| Test: используется только в конце")

low_sample, high_sample = next(iter(restore_val_loader))
assert low_sample.shape[1:] == (3, 16, 16)
assert high_sample.shape[1:] == (3, 32, 32)

In [ ]:
# === РЕШЕНИЕ ===
def bicubic_upscale(low):
    return F.interpolate(low, size=(32, 32), mode="bicubic", align_corners=False).clamp(0, 1)


def psnr_per_image(pred, target):
    mse = ((pred.clamp(0, 1) - target) ** 2).flatten(1).mean(1)
    return 10 * torch.log10(1.0 / mse)


def show_restoration(low, pred, high, n=5, title=""):
    low, pred, high = low[:n].cpu(), pred[:n].detach().cpu().clamp(0, 1), high[:n].cpu()
    ps = psnr_per_image(pred, high)
    fig, axes = plt.subplots(3, n, figsize=(2.2 * n, 7))
    for i in range(n):
        for r, img in enumerate((high[i], low[i], pred[i])):
            axes[r, i].imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest")
            axes[r, i].axis("off")
        axes[2, i].set_title(("восстановление, " if i == 0 else "") + f"PSNR {ps[i]:.1f} дБ", fontsize=9)
    axes[0, 0].set_title("исходное 32x32", fontsize=9)
    axes[1, 0].set_title("вход 16x16", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def mse_on_loader(model, loader):
    if isinstance(model, nn.Module):
        model.eval()
    total, count = 0.0, 0
    for low, high in loader:
        low, high = low.to(device), high.to(device)
        total += F.mse_loss(model(low).clamp(0, 1), high, reduction="sum").item()
        count += high.numel()
    return total / count


to_psnr = lambda mse: 10 * np.log10(1.0 / mse)

show_restoration(low_sample, bicubic_upscale(low_sample), high_sample, n=6,
                 title="Validation: исходное, уменьшенное 16x16, bicubic 32x32")
bicubic_val_mse = mse_on_loader(bicubic_upscale, restore_val_loader)
print(f"Bicubic на validation: MSE = {bicubic_val_mse:.5f}, PSNR = {to_psnr(bicubic_val_mse):.2f} дБ")

В чем различие между downsampling, интерполяцией и нейросетевым восстановлением? Почему пары low/high нельзя распределять по train/val отдельно? 

Что модель не может знать точно. При уменьшении 32x32 -> 16x16 (`mode="area"`) каждый блок 2x2 заменяется средним: из четырех значений остается одно, 3/4 информации теряется. Мелкие детали, текстура, тонкие линии и точное положение резкой границы внутри блока однозначно восстановить нельзя: одному изображению 16x16 соответствует бесконечно много изображений 32x32. Модель может только угадать правдоподобные детали по статистике обучающих данных.

Downsampling - уменьшение разрешения с потерей информации (здесь усреднение блоков). Интерполяция (bilinear, bicubic) - фиксированная формула, которая распределяет известные пиксели по новой сетке; результат гладкий, новых деталей нет. Нейросетевое восстановление - обученное отображение low -> high: по примерам сеть запоминает типичные края и текстуры и в среднем восстанавливает их лучше интерполяции, но это догадка, а не настоящая информация.

Почему пары нельзя делить по train/val отдельно. Пример - это пара из двух версий одной картинки. Если делить их независимо, вход перестанет соответствовать цели, а одна и та же картинка может попасть в train как high и в val как low. Это утечка данных, и валидация покажет завышенное качество. Поэтому сначала делятся индексы картинок, а пары строятся внутри каждой части, как в `LowHighPairs`.

### 2.2. Компактный автоэнкодер для super-resolution x2

Стартовая модель принимает цветное 16 x 16 изображение и выдаёт 32 x 32 изображение. Для простоты оптимизируем **MSE**. Сразу проверяем формы тензоров: ошибочная размерность часто приводит к неочевидному broadcasting в loss.


In [ ]:
class SkipEncoder(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(width, 2 * width, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(2 * width, 2 * width, 3, padding=1), nn.ReLU(inplace=True))

    def forward(self, x):
        return torch.cat([x, self.body(x)], dim=1)


class UpsampleDecoder(nn.Module):
    def __init__(self, width, residual=True):
        super().__init__()
        self.residual = residual
        self.body = nn.Sequential(nn.Conv2d(2 * width, 3 * 4, 3, padding=1), nn.PixelShuffle(2))

    def forward(self, h):
        x, feats = h[:, :3], h[:, 3:]
        out = self.body(feats)
        if self.residual:
            return F.interpolate(x, size=(32, 32), mode="bicubic", align_corners=False) + out
        return torch.sigmoid(out)


class SuperResolutionAE(nn.Module):
    def __init__(self, width=32, residual=True):
        # === РЕШЕНИЕ ===
        super().__init__()
        self.encoder = SkipEncoder(width)
        self.decoder = UpsampleDecoder(width, residual)

    def forward(self, x):
        return self.decoder(self.encoder(x))

AE = SuperResolutionAE().to(device)
with torch.no_grad():
    ae_check = AE(torch.rand(2, 3, 16, 16, device=device))
assert ae_check.shape == (2, 3, 32, 32), ae_check.shape
print("AE parameters:", count_params(AE), "| output:", tuple(ae_check.shape))

### 2.3. Обучение по MSE + выбор лучшей эпохи на validation

Для каждого batch предсказываем high-resolution по low-resolution, оптимизируем MSE. После эпохи смотрим на **validation**, сохраняем копию лучшей модели. **Test не трогаем при выборе гиперпараметров и checkpoint.**

In [ ]:
# === РЕШЕНИЕ ===
def train_restorer(model, epochs=RESTORE_EPOCHS, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train_mse": [], "val_mse": []}
    best_val, best_state, best_epoch = float("inf"), None, 0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        total, count = 0.0, 0
        for low, high in restore_train_loader:
            low, high = low.to(device), high.to(device)
            pred = model(low)
            assert pred.shape == high.shape, (pred.shape, high.shape)
            loss = F.mse_loss(pred, high)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * high.numel()
            count += high.numel()
        val_mse = mse_on_loader(model, restore_val_loader)
        hist["train_mse"].append(total / count)
        hist["val_mse"].append(val_mse)
        if val_mse < best_val:
            best_val, best_state, best_epoch = val_mse, copy.deepcopy(model.state_dict()), epoch
        print(f"  эпоха {epoch}/{epochs} | train PSNR {to_psnr(total / count):.2f} | val PSNR {to_psnr(val_mse):.2f} дБ"
              f"{'  (лучшая)' if best_epoch == epoch else ''} | {time.time() - t0:.0f} c")
    model.load_state_dict(best_state)
    hist["best_epoch"], hist["time"] = best_epoch, time.time() - t0
    return model, hist


print("Автоэнкодер со skip-связью (предсказывает поправку к bicubic):")
torch.manual_seed(SEED)
AE = SuperResolutionAE(residual=True).to(device)
AE, ae_hist = train_restorer(AE)

print("\nТот же автоэнкодер без skip-связи (строит картинку целиком):")
torch.manual_seed(SEED)
AE_plain = SuperResolutionAE(residual=False).to(device)
AE_plain, plain_hist = train_restorer(AE_plain)

fig, ax = plt.subplots(figsize=(8, 4.2))
ep = np.arange(1, RESTORE_EPOCHS + 1)
for h, name, color in [(ae_hist, "AE + skip", "#2a78d6"), (plain_hist, "AE без skip", "#eb6834")]:
    ax.plot(ep, to_psnr(np.array(h["train_mse"])), "--", color=color, lw=1.5, label=f"{name}: train")
    ax.plot(ep, to_psnr(np.array(h["val_mse"])), "o-", color=color, label=f"{name}: val")
ax.axhline(to_psnr(bicubic_val_mse), color="0.4", ls=":", label="bicubic: val")
ax.set_xlabel("эпоха")
ax.set_ylabel("PSNR, дБ")
ax.set_title("Обучение super-resolution x2")
ax.legend(fontsize=8)
plt.show()
print(f"Лучшая эпоха по validation: AE + skip - {ae_hist['best_epoch']}, AE без skip - {plain_hist['best_epoch']}")

### 2.4. Итоговая оценка на отложенном test

Теперь один раз загружаем официальный test и считаем метрики двух методов на **одинаковых изображениях**:

- `MSE` - средний квадрат ошибки (меньше - лучше).
- `PSNR = 10 log10(1/MSE)` для изображений в `[0,1]` (больше - лучше).

PSNR не равен перцептивному качеству. Более "резкая" картинка может иметь худший PSNR, а хорошая MSE не гарантирует реалистичность деталей. Выведите не менее 5 триплетов и найдите **один пример, где нейросеть выглядит лучше, и один - где хуже либо не имеет явного преимущества**. Если положительного примера нет, так и напишите.

In [ ]:
cifar_test = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True,
                              transform=transforms.ToTensor())
restore_test_loader = DataLoader(LowHighPairs(cifar_test), batch_size=BATCH_SIZE,
                                 shuffle=False, num_workers=NUM_WORKERS)

@torch.no_grad()
def evaluate_restoration(model, loader):
    model.eval()
    total_ae = total_bicubic = 0.0
    total_pixels = 0
    for low, high in loader:
        low, high = low.to(device), high.to(device)
        ae_pred = model(low).clamp(0, 1)
        bicubic_pred = bicubic_upscale(low)
        total_ae += F.mse_loss(ae_pred, high, reduction="sum").item()
        total_bicubic += F.mse_loss(bicubic_pred, high, reduction="sum").item()
        total_pixels += high.numel()
    ae_mse = total_ae / total_pixels
    bic_mse = total_bicubic / total_pixels
    psnr = lambda mse: float("inf") if mse == 0 else 10 * np.log10(1.0 / mse)
    return {"Bicubic": (bic_mse, psnr(bic_mse)), "Autoencoder": (ae_mse, psnr(ae_mse))}

final_metrics = evaluate_restoration(AE, restore_test_loader)
print(f"{'Method':<15} {'MSE ':>12} {'PSNR ':>12}")
for name, (mse, psnr) in final_metrics.items():
    print(f"{name:<15} {mse:12.6f} {psnr:12.3f}")

low_test, high_test = next(iter(restore_test_loader))
with torch.no_grad():
    ae_test = AE(low_test.to(device)).cpu()
show_restoration(low_test, bicubic_upscale(low_test), high_test, n=5,
                 title="Test: bicubic")
show_restoration(low_test, ae_test, high_test, n=5,
                 title="Test: autoencoder")

In [ ]:
# === РЕШЕНИЕ (продолжение 2.4): PSNR по отдельным картинкам на всем test ===
@torch.no_grad()
def per_image_psnr(model, loader):
    model.eval()
    p_bic, p_model = [], []
    for low, high in loader:
        low, high = low.to(device), high.to(device)
        p_bic.append(psnr_per_image(bicubic_upscale(low), high).cpu())
        p_model.append(psnr_per_image(model(low), high).cpu())
    return torch.cat(p_bic).numpy(), torch.cat(p_model).numpy()


psnr_bic, psnr_ae = per_image_psnr(AE, restore_test_loader)
gain = psnr_ae - psnr_bic
plain_metrics = evaluate_restoration(AE_plain, restore_test_loader)["Autoencoder"]
print(f"AE без skip-связи на test: MSE {plain_metrics[0]:.6f}, PSNR {plain_metrics[1]:.3f} дБ")
print(f"AE лучше bicubic на {np.mean(gain > 0):.1%} test-картинок, выигрыш PSNR: среднее {gain.mean():+.2f} дБ, "
      f"медиана {np.median(gain):+.2f} дБ, min {gain.min():+.2f}, max {gain.max():+.2f}")

test_pairs = LowHighPairs(cifar_test)
best_i, worst_i = int(np.argmax(gain)), int(np.argmin(gain))
fig, axes = plt.subplots(2, 5, figsize=(14, 6.2))
for row, (i, label) in enumerate([(best_i, "нейросеть лучше"), (worst_i, "нейросеть не лучше")]):
    low, high = test_pairs[i]
    with torch.no_grad():
        ae_img = AE(low.unsqueeze(0).to(device)).cpu().clamp(0, 1)[0]
    bic_img = bicubic_upscale(low.unsqueeze(0))[0]
    panels = [(high, f"{label}\nисходное"), (low, "вход 16x16"), (bic_img, f"bicubic {psnr_bic[i]:.1f} дБ"),
              (ae_img, f"AE {psnr_ae[i]:.1f} дБ"),
              ((ae_img - high).abs().mean(0, keepdim=True).repeat(3, 1, 1) * 4, "|ошибка AE| x4")]
    for ax, (img, t) in zip(axes[row], panels):
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy(), interpolation="nearest")
        ax.axis("off")
        ax.set_title(t, fontsize=9)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(gain, bins=60, color="#2a78d6")
ax.axvline(0, color="k", lw=1)
ax.set_xlabel("PSNR(AE) - PSNR(bicubic), дБ")
ax.set_ylabel("число картинок")
ax.set_title("Выигрыш нейросети по отдельным test-картинкам")
plt.show()

detail = torch.stack([(lambda h: (h[:, 1:, :] - h[:, :-1, :]).abs().mean() + (h[:, :, 1:] - h[:, :, :-1]).abs().mean())(
    test_pairs[i][1]) for i in range(len(test_pairs))]).numpy()
q = np.quantile(detail, [0.2, 0.8])
gain_smooth, gain_detail = gain[detail <= q[0]].mean(), gain[detail >= q[1]].mean()
print(f"Средний выигрыш AE: на 20% самых гладких картинок {gain_smooth:+.2f} дБ, на 20% самых детальных {gain_detail:+.2f} дБ")

#### Ответ по 2.4

In [ ]:
bic_mse, bic_psnr = final_metrics["Bicubic"]
ae_mse, ae_psnr = final_metrics["Autoencoder"]
text = [f"- На test ({len(cifar_test)} картинок): bicubic PSNR {bic_psnr:.2f} дБ (MSE {bic_mse:.5f}), "
        f"автоэнкодер PSNR {ae_psnr:.2f} дБ (MSE {ae_mse:.5f})."]
if ae_psnr > bic_psnr:
    text.append(f"- Автоэнкодер лучше на {ae_psnr - bic_psnr:.2f} дБ, ошибка меньше на {(1 - ae_mse / bic_mse):.0%}. "
                f"Он лучше bicubic на {np.mean(gain > 0):.1%} картинок, медианный выигрыш {np.median(gain):+.2f} дБ.")
else:
    text.append(f"- Автоэнкодер не обогнал bicubic (разница {ae_psnr - bic_psnr:+.2f} дБ), "
                f"лучше он только на {np.mean(gain > 0):.1%} картинок.")
if gain[best_i] > 0:
    text.append(f"- Пример, где нейросеть лучше: выигрыш {gain[best_i]:+.2f} дБ.")
else:
    text.append("- Положительного примера нет: ни на одной test-картинке AE не обогнал bicubic.")
text.append(f"- Пример, где нейросеть не лучше: {gain[worst_i]:+.2f} дБ."
            + (" Там, где догадка сети о краях неверная, ошибка выше, чем у гладкой интерполяции."
               if gain[worst_i] < 0 else ""))
text.append(f"- Без skip-связи тот же автоэнкодер получил {plain_metrics[1]:.2f} дБ"
            + (f", это хуже bicubic на {bic_psnr - plain_metrics[1]:.2f} дБ. " if plain_metrics[1] < bic_psnr
               else f", это выше bicubic на {plain_metrics[1] - bic_psnr:.2f} дБ. ")
            + "Без skip-связи сети приходится учиться воспроизводить всю картинку, включая цвета, а со skip-связью она "
            "учит только недостающие мелкие детали.")
text.append(f"- Выигрыш на гладких картинках {gain_smooth:+.2f} дБ, на детальных {gain_detail:+.2f} дБ.")
text.append("- PSNR не равен визуальному качеству: при MSE в неясных местах выгодно предсказывать среднее, а среднее "
            "размыто, поэтому обе картинки выглядят мягкими, даже когда PSNR у сети выше.")
display(Markdown('\n'.join(text)))

### 2.5. Дополнительно: условный GAN для восстановления

Генератор - тот же автоэнкодер, начинаем с весов лучшей модели из 2.3. Условный дискриминатор (PatchGAN) получает пару "вход (bicubic) + кандидат 32x32" и для каждого участка решает, похож ли он на настоящий. Loss генератора = MSE + 0.001 * adversarial, как в SRGAN.

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self, ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(6, ch, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ch, 2 * ch, 4, 2, 1), nn.BatchNorm2d(2 * ch), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(2 * ch, 1, 4, 1, 1))

    def forward(self, low, img):
        return self.net(torch.cat([bicubic_upscale(low), img], dim=1))


LAMBDA_ADV = 1e-3
torch.manual_seed(SEED)
G_sr = copy.deepcopy(AE)
D_sr = PatchDiscriminator().to(device)
opt_gs = torch.optim.Adam(G_sr.parameters(), lr=1e-4, betas=(0.5, 0.999))
opt_ds = torch.optim.Adam(D_sr.parameters(), lr=1e-4, betas=(0.5, 0.999))
print("D_sr parameters:", count_params(D_sr))
cgan_hist = {"val_psnr": [], "d_loss": [], "adv": []}
t0 = time.time()
for epoch in range(1, RESTORE_EPOCHS + 1):
    G_sr.train()
    D_sr.train()
    d_losses, advs = [], []
    for low, high in restore_train_loader:
        low, high = low.to(device), high.to(device)
        fake = G_sr(low)
        d_real, d_fake = D_sr(low, high), D_sr(low, fake.detach())
        loss_d = criterion_adv(d_real, torch.ones_like(d_real)) + criterion_adv(d_fake, torch.zeros_like(d_fake))
        opt_ds.zero_grad()
        loss_d.backward()
        opt_ds.step()
        d_fake = D_sr(low, fake)
        adv = criterion_adv(d_fake, torch.ones_like(d_fake))
        loss_g = F.mse_loss(fake, high) + LAMBDA_ADV * adv
        opt_gs.zero_grad()
        loss_g.backward()
        opt_gs.step()
        d_losses.append(loss_d.item())
        advs.append(adv.item())
    cgan_hist["val_psnr"].append(to_psnr(mse_on_loader(G_sr, restore_val_loader)))
    cgan_hist["d_loss"].append(np.mean(d_losses))
    cgan_hist["adv"].append(np.mean(advs))
    print(f"  эпоха {epoch}/{RESTORE_EPOCHS} | D loss {cgan_hist['d_loss'][-1]:.3f} | adv {cgan_hist['adv'][-1]:.3f} | "
          f"val PSNR {cgan_hist['val_psnr'][-1]:.2f} дБ | {time.time() - t0:.0f} c")


def sharpness(x):
    return ((x[..., 1:, :] - x[..., :-1, :]).abs().mean() + (x[..., :, 1:] - x[..., :, :-1]).abs().mean()).item()


cgan_metrics = evaluate_restoration(G_sr, restore_test_loader)["Autoencoder"]
low_t, high_t = next(iter(restore_test_loader))
with torch.no_grad():
    preds = {"bicubic": bicubic_upscale(low_t), "AE (MSE)": AE(low_t.to(device)).cpu().clamp(0, 1),
             "cGAN": G_sr(low_t.to(device)).cpu().clamp(0, 1)}
sharp = {k: sharpness(v) for k, v in preds.items()}
sharp_true = sharpness(high_t)
print(f"\n{'Метод':<10} {'MSE':>10} {'PSNR':>9} {'Резкость':>10}  (резкость исходных картинок: {sharp_true:.4f})")
for name, (mse, ps) in [("bicubic", final_metrics["Bicubic"]), ("AE (MSE)", final_metrics["Autoencoder"]),
                        ("cGAN", cgan_metrics)]:
    print(f"{name:<10} {mse:10.6f} {ps:9.3f} {sharp[name]:10.4f}")

fig, axes = plt.subplots(4, 6, figsize=(12, 8.4))
for i in range(6):
    for r, (img, t) in enumerate([(high_t[i], "исходное"), (preds["bicubic"][i], "bicubic"),
                                  (preds["AE (MSE)"][i], "AE (MSE)"), (preds["cGAN"][i], "cGAN")]):
        axes[r, i].imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest")
        axes[r, i].axis("off")
        if i == 0:
            axes[r, i].set_title(t, fontsize=9, loc="left")
fig.suptitle("Test: сравнение методов восстановления")
plt.tight_layout()
plt.show()

#### Ответ по 2.5

In [ ]:
ae_psnr = final_metrics["Autoencoder"][1]
text = [f"- Test PSNR: cGAN {cgan_metrics[1]:.2f} дБ, AE {ae_psnr:.2f} дБ, bicubic {final_metrics['Bicubic'][1]:.2f} дБ.",
        f"- Резкость (средний модуль градиента): cGAN {sharp['cGAN']:.4f}, AE {sharp['AE (MSE)']:.4f}, "
        f"bicubic {sharp['bicubic']:.4f}, у исходных картинок {sharp_true:.4f}."]
if sharp['cGAN'] > sharp['AE (MSE)'] and cgan_metrics[1] < ae_psnr:
    text.append("- Состязательная часть сделала картинки немного резче ценой небольшой потери PSNR. Это обычный "
                "компромисс между точностью по пикселям и правдоподобием деталей.")
elif cgan_metrics[1] >= ae_psnr:
    text.append("- cGAN не уступил автоэнкодеру по PSNR, дополнительное обучение пошло на пользу.")
else:
    text.append("- Заметного выигрыша в резкости cGAN не дал.")
text.append(f"- Val PSNR по эпохам: {', '.join(f'{v:.2f}' for v in cgan_hist['val_psnr'])}; D loss: "
            f"{', '.join(f'{v:.2f}' for v in cgan_hist['d_loss'])}"
            + (". D loss снижается, дискриминатор учится отличать восстановленные картинки от настоящих."
               if cgan_hist['d_loss'][-1] < cgan_hist['d_loss'][0] - 0.05 else "."))
text.append(f"- Вклад adversarial части маленький (lambda = {LAMBDA_ADV}), поэтому визуально cGAN близок к AE. "
            "Если увеличить lambda, картинка станет резче, но PSNR упадет и могут появиться ложные текстуры.")
display(Markdown('\n'.join(text)))

## Итоговые выводы

In [ ]:
text = [
    f"1. GAN на Fashion-MNIST (эпох: {GAN_EPOCHS}, картинок: {len(gan_data)}): классов с долей от 2% среди "
    f"сгенерированных - {len(present)} из 10, разнообразие {div_gen:.1f} против {div_real:.1f} у реальных картинок. В последней эпохе "
    f"D(x) = {ep_real[-1]:.2f}, D(G(z)) = {ep_fake[-1]:.2f}. Мелкий шум (модуль лапласиана): "
    f"{noise_gen:.3f} у сгенерированных против {noise_real:.3f} у реальных картинок.",
    ("2. Латентное пространство гладкое, интерполяция дает плавные переходы между предметами. "
     if step_ratio < 2.5 else "2. При интерполяции в латентном пространстве встречаются скачки. ")
    + f"При масштабировании z класс предмета сохранился у {same_cls} из 8 векторов. "
    "Судить о качестве можно только по z из распределения обучения.",
    f"3. Восстановление 16x16 -> 32x32: bicubic {final_metrics['Bicubic'][1]:.2f} дБ, автоэнкодер со skip-связью "
    f"{final_metrics['Autoencoder'][1]:.2f} дБ, без skip-связи {plain_metrics[1]:.2f} дБ."
    + (" Главное в архитектуре - учить поправку к интерполяции, а не всю картинку."
       if plain_metrics[1] < final_metrics['Autoencoder'][1] else ""),
    f"4. cGAN: {cgan_metrics[1]:.2f} дБ, резкость {sharp['cGAN']:.4f} против {sharp['AE (MSE)']:.4f} у AE. "
    "Метрики искажения (MSE, PSNR) и визуальное качество - разные цели.",
]
display(Markdown('\n'.join(text)))